# Clustering OASIS-1 Brain MRI Patients — Results Summary

---

## Project Overview

This notebook summarises the results of five unsupervised clustering experiments
applied to the **OASIS-1 cross-sectional MRI dataset**.

The central question is: *can we recover meaningful subgroups of patients
based on brain MRI structure and/or clinical variables, without using the
dementia severity rating (CDR) as a label?*

---

### Dataset

| Property | Value |
|----------|-------|
| Patients | 416 |
| MRI image types per patient | 5 (coronal, sagittal GFC, transverse GFC, masked transverse, subject-space sagittal) |
| Clinical variables available | Age, Sex, Education, SES, MMSE, brain volumes (eTIV, nWBV, ASF) |
| Evaluation variable (not used in clustering) | CDR (0 = no dementia, 0.5 = very mild, 1 = mild, 2 = moderate) |
| Patients with CDR label | 235 / 416 |

All methods use **k = 4 clusters** and **FasterPAM k-medoids** to align with
the four CDR levels.  CDR is loaded after clustering solely to evaluate whether
the discovered groups align with dementia severity.

---

## Methods Tested

| # | Method name | Features used | Distance measure | Feature dimensions |
|---|-------------|---------------|-----------------|--------------------|
| 1 | HOG + Global PCA | HOG descriptors → global PCA (50 components) | Euclidean | 50 |
| 2 | Clinical only | Age, Sex, Education, SES, MMSE, eTIV, nWBV, ASF | Generalised Gower (Minkowski + Sokal) | 8 |
| 3 | Per-image PCA | Separate PCA per image type at 59.2 % variance → concatenated | Robust Mahalanobis | 476 |
| 4 | Clinical + Global PCA | Clinical (8) + global PCA (50) together | Generalised Gower | 58 |
| 5 | Clinical + Per-image PCA | Clinical (8) + per-image PCA (476) together | Generalised Gower | 484 |

**Notes on pre-processing:** `Delay` (100 % missing) and `Hand` (constant = Right)
were dropped before clustering in all methods that include clinical features.
`Education`, `SES`, and `MMSE` were imputed with their column medians (~43–48 % missing).

In [1]:
import os
import numpy as np
import pandas as pd

NB_DIR = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks"

METHODS = {
    "HOG + Global PCA\n(Euclidean)": "method2_hog_pca_global_results.csv",
    "Clinical only\n(Gen. Gower)": "clinical_clustering_results.csv",
    "Per-image PCA\n(Rob. Mahalanobis)": "pca_per_image_clustering_results.csv",
    "Clinical + Global PCA\n(Gen. Gower)": "combined_clinical_pca_global_results.csv",
    "Clinical + Per-image PCA\n(Gen. Gower)": "combined_clinical_pca_per_image_results.csv",
}

def load_result(filename):
    path = os.path.join(NB_DIR, filename)
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return df

def crosstab_pct(df):
    """Row-normalised crosstab of cluster vs CDR for patients with CDR."""
    has_cdr = df[df["CDR"].notnull()].copy()
    has_cdr["CDR"] = has_cdr["CDR"].astype(str)
    ct_pct = pd.crosstab(has_cdr["cluster"], has_cdr["CDR"], normalize="index").mul(100).round(1)
    return ct_pct

def crosstab_counts(df):
    """Count crosstab of cluster vs CDR."""
    has_cdr = df[df["CDR"].notnull()].copy()
    has_cdr["CDR"] = has_cdr["CDR"].astype(str)
    return pd.crosstab(has_cdr["cluster"], has_cdr["CDR"],
                       margins=True, margins_name="Total")

def gradient_label(spread):
    if spread >= 40: return "Yes (strong)"
    if spread >= 18: return "Partial"
    return "No"

print("Helper functions loaded. Result files found:")
for name, fname in METHODS.items():
    status = "OK" if os.path.exists(os.path.join(NB_DIR, fname)) else "MISSING"
    print(f"  [{status}]  {fname}")

Helper functions loaded. Result files found:
  [OK]  method2_hog_pca_global_results.csv
  [OK]  clinical_clustering_results.csv
  [OK]  pca_per_image_clustering_results.csv
  [OK]  combined_clinical_pca_global_results.csv
  [OK]  combined_clinical_pca_per_image_results.csv


---

## Comparison Table — All Methods

The table below summarises the key clustering quality metric: **how well do the
clusters separate CDR = 0 (non-demented) patients from impaired ones?**

- **Best cluster (% CDR = 0)** — the cluster with the highest proportion of healthy patients
- **Worst cluster (% CDR = 0)** — the cluster with the lowest proportion of healthy patients
- **Spread** — difference between best and worst; larger = better separation
- **Gradient** — whether there is a meaningful progressive ordering across clusters

In [2]:
rows = []
for name, fname in METHODS.items():
    df = load_result(fname)
    if df is None:
        rows.append({"Method": name.replace("\n", " "), "Features": "—",
                     "Distance": "—", "Best CDR=0 %": "N/A",
                     "Worst CDR=0 %": "N/A", "Spread (pp)": "N/A",
                     "Gradient": "N/A"})
        continue

    ct = crosstab_pct(df)
    if "0.0" not in ct.columns:
        continue
    best  = ct["0.0"].max()
    worst = ct["0.0"].min()
    spread = best - worst

    rows.append({
        "Method"        : name.replace("\n", " "),
        "Best CDR=0 %"  : f"{best:.1f} %",
        "Worst CDR=0 %" : f"{worst:.1f} %",
        "Spread (pp)"   : f"{spread:.1f}",
        "Gradient"      : gradient_label(spread),
    })

summary = pd.DataFrame(rows)
summary.index = range(1, len(summary) + 1)
summary.index.name = "#"
print("Clustering performance across all methods (CDR = 0 separation):")
summary

Clustering performance across all methods (CDR = 0 separation):


,Method,Best CDR=0 %,Worst CDR=0 %,Spread (pp),Gradient
#,,,,,
1,HOG + Global PCA (Euclidean),93.3 %,45.0 %,48.3,Yes (strong)
2,Clinical only (Gen. Gower),66.7 %,43.6 %,23.1,Partial
3,Per-image PCA (Rob. Mahalanobis),65.1 %,44.6 %,20.5,Partial
4,Clinical + Global PCA (Gen. Gower),67.7 %,43.6 %,24.1,Partial
5,Clinical + Per-image PCA (Gen. Gower),67.4 %,45.7 %,21.7,Partial


---

## Individual Method Results

Each section below shows: cluster sizes, the count crosstab, the row-normalised
percentage table, and a written interpretation.

### Method 1 — HOG + Global PCA (Euclidean distance)

**Features:** HOG descriptors extracted from all 5 MRI images per patient,
averaged into one vector, then reduced to 50 components via a single global PCA.  
**Distance:** Euclidean on the 50 PCA components.  
**Key parameters:** TARGET_SIZE = 128×128, orientations = 9,
pixels_per_cell = 8×8, cells_per_block = 2×2, PCA components = 50.

In [3]:
df1 = load_result("method2_hog_pca_global_results.csv")

print("Cluster sizes:")
print(df1["cluster"].value_counts().sort_index().rename("count").to_string())
print()
print("Cluster × CDR (counts):")
display(crosstab_counts(df1))
print("Row-normalised (% of each CDR level within cluster):")
crosstab_pct(df1)

Cluster sizes:
cluster
0.0     78
1.0     77
2.0    112
3.0    149

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0.0,14,1,0,0,15
1.0,21,4,0,0,25
2.0,37,12,5,1,55
3.0,63,53,23,1,140
Total,135,70,28,2,235


Row-normalised (% of each CDR level within cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0.0,93.3,6.7,0.0,0.0
1.0,84.0,16.0,0.0,0.0
2.0,67.3,21.8,9.1,1.8
3.0,45.0,37.9,16.4,0.7


**Interpretation:** This method produces the strongest CDR separation of all five approaches, with a 48.3 percentage point spread between the healthiest cluster (93.3 % CDR = 0) and the most impaired one (45.0 % CDR = 0). The clusters form a clear gradient — moving from cluster 0 (almost entirely non-demented) through to cluster 3, which contains the highest concentration of CDR ≥ 0.5 patients. This suggests that averaging HOG features across all five image planes and projecting onto a common PCA space captures structural brain variation that correlates meaningfully with cognitive decline.

### Method 2 — Clinical Variables Only (Generalised Gower distance)

**Features:** Eight clinical variables — Age, Education, SES, MMSE, eTIV, nWBV, ASF, and Sex (binary).  
**Distance:** Generalised Gower (Manhattan for continuous/ordinal, Sokal for binary).  
**Key parameters:** Education, SES, MMSE imputed with column median; Hand and Delay dropped.

In [4]:
df2 = load_result("clinical_clustering_results.csv")

print("Cluster sizes:")
print(df2["cluster"].value_counts().sort_index().rename("count").to_string())
print()
print("Cluster × CDR (counts):")
display(crosstab_counts(df2))
print("Row-normalised (% of each CDR level within cluster):")
crosstab_pct(df2)

Cluster sizes:
cluster
0    128
1    128
2     76
3     84

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,35,17,11,0,63
1,62,22,8,1,93
2,21,14,5,0,40
3,17,17,4,1,39
Total,135,70,28,2,235


Row-normalised (% of each CDR level within cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,55.6,27.0,17.5,0.0
1,66.7,23.7,8.6,1.1
2,52.5,35.0,12.5,0.0
3,43.6,43.6,10.3,2.6


**Interpretation:** Clinical variables alone produce only a 23.1 percentage point spread (43.6 % to 66.7 % CDR = 0), with no consistent ordering across clusters — the cluster with the highest healthy proportion is not clearly separated from the others. This confirms that demographic and cognitive measures overlap substantially across CDR levels in OASIS-1, making it difficult to discover dementia-severity subgroups from clinical data alone. The Generalised Gower distance handles the mixed variable types correctly, but the signal in the features is insufficient for strong separation.

### Method 3 — Per-Image PCA (Robust Mahalanobis distance)

**Features:** A separate PCA was fitted on each of the 5 image types, retaining
enough components to explain ≥ 59.2 % of variance per type (110, 105, 107, 112, 42 components respectively). All 476 components were concatenated into one vector per patient.  
**Distance:** Robust Mahalanobis (trimmed covariance, α = 0.05).  
**Key parameters:** Variance threshold = 59.2 %, total features = 476.

In [5]:
df3 = load_result("pca_per_image_clustering_results.csv")

print("Cluster sizes:")
print(df3["cluster"].value_counts().sort_index().rename("count").to_string())
print()
print("Cluster × CDR (counts):")
display(crosstab_counts(df3))
print("Row-normalised (% of each CDR level within cluster):")
crosstab_pct(df3)

Cluster sizes:
cluster
0    101
1    142
2     94
3     79

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,25,21,8,2,56
1,47,20,10,0,77
2,35,18,6,0,59
3,28,11,4,0,43
Total,135,70,28,2,235


Row-normalised (% of each CDR level within cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,44.6,37.5,14.3,3.6
1,61.0,26.0,13.0,0.0
2,59.3,30.5,10.2,0.0
3,65.1,25.6,9.3,0.0


**Interpretation:** The per-image PCA approach achieves the smallest spread of the three MRI-based methods (20.5 pp), performing no better than clinical variables alone. Treating each image type independently fragments the structural information that the global approach preserves — the global PCA sees all images together and can discover cross-plane variation, while the per-image approach cannot. The high dimensionality (476 features) combined with Robust Mahalanobis distance may also amplify noise rather than the signal relevant to cognitive decline.

### Method 4 — Clinical + Global PCA Combined (Generalised Gower distance)

**Features:** Eight clinical variables combined with the 50 global PCA components
from Method 1 — all processed together in a single distance computation.  
**Distance:** Generalised Gower (p1 = 57 quantitative, p2 = 1 binary).  
**Key parameters:** Same clinical pre-processing as Method 2; same PCA as Method 1.

In [6]:
df4 = load_result("combined_clinical_pca_global_results.csv")

print("Cluster sizes:")
print(df4["cluster"].value_counts().sort_index().rename("count").to_string())
print()
print("Cluster × CDR (counts):")
display(crosstab_counts(df4))
print("Row-normalised (% of each CDR level within cluster):")
crosstab_pct(df4)

Cluster sizes:
cluster
0    122
1    134
2     76
3     84

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,32,17,11,0,60
1,65,22,8,1,96
2,21,14,5,0,40
3,17,17,4,1,39
Total,135,70,28,2,235


Row-normalised (% of each CDR level within cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,53.3,28.3,18.3,0.0
1,67.7,22.9,8.3,1.0
2,52.5,35.0,12.5,0.0
3,43.6,43.6,10.3,2.6


**Interpretation:** Combining clinical variables with the 50 global PCA components yields a spread of 24.1 pp — a marginal improvement over clinical alone (23.1 pp), but far below the MRI-only result (48.3 pp). Rather than adding the strengths of both modalities, the combination appears to dilute the strong MRI signal: the Generalised Gower distance normalises each feature's contribution, so the 50 PCA dimensions are averaged against the 8 clinical variables and their discriminative power is reduced. This suggests that MRI features are best kept separate from clinical variables for this task.

### Method 5 — Clinical + Per-Image PCA Combined (Generalised Gower distance)

**Features:** Eight clinical variables combined with all 476 per-image PCA components
from Method 3 — 484 features total processed together.  
**Distance:** Generalised Gower (p1 = 483 quantitative, p2 = 1 binary).  
**Key parameters:** Same clinical pre-processing as Method 2; same per-image PCA as Method 3.

In [7]:
df5 = load_result("combined_clinical_pca_per_image_results.csv")

print("Cluster sizes:")
print(df5["cluster"].value_counts().sort_index().rename("count").to_string())
print()
print("Cluster × CDR (counts):")
display(crosstab_counts(df5))
print("Row-normalised (% of each CDR level within cluster):")
crosstab_pct(df5)

Cluster sizes:
cluster
0    117
1    139
2     63
3     97

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,33,17,11,0,61
1,64,22,8,1,95
2,17,12,4,0,33
3,21,19,5,1,46
Total,135,70,28,2,235


Row-normalised (% of each CDR level within cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,54.1,27.9,18.0,0.0
1,67.4,23.2,8.4,1.1
2,51.5,36.4,12.1,0.0
3,45.7,41.3,10.9,2.2


**Interpretation:** Adding 476 per-image PCA components to the clinical variables produces a spread of 21.7 pp, nearly identical to using clinical features alone (23.1 pp) and slightly worse than Method 4. With 484 dimensions dominated by MRI components that individually carry weak signal, the distance measure is largely driven by noise, leaving the clustering no better than a clinical baseline. This reinforces the finding from Method 3: independently computed per-image PCA features do not add useful discriminative information when combined with clinical data.

---

## Conclusions

Across five clustering experiments on the OASIS-1 dataset, the following key findings emerge:

**1. Global MRI features outperform all other approaches.**  
Method 1 (HOG + Global PCA, Euclidean) achieves a 48.3 percentage point spread in CDR = 0 proportions across clusters — more than twice the spread of every other method. Averaging HOG features across all five image planes and reducing them jointly with a single PCA preserves cross-plane structural information that directly correlates with cognitive decline.

**2. Clinical variables alone are insufficient.**  
Methods 2, 4, and 5 all produce spreads between 21–24 pp with no clear CDR gradient. Age, education, MMSE, and brain volume measurements overlap too broadly across dementia severity levels to drive meaningful unsupervised separation in this dataset.

**3. Combining modalities does not help.**  
Methods 4 and 5 show that merging clinical and MRI features into a single Generalised Gower distance matrix dilutes rather than amplifies the MRI signal. The distance normalisation averages the contribution of each feature, causing the strong MRI signal from Method 1 to be offset by the weaker clinical signal.

**4. Per-image PCA underperforms global PCA.**  
Fitting separate PCAs per image type (Method 3) and concatenating them sacrifices the cross-plane structural correlation that the global approach exploits. The resulting 476-dimensional space with Robust Mahalanobis distance performs no better than clinical variables alone.

**Overall recommendation:**  
For unsupervised clustering of OASIS-1 patients, the HOG + Global PCA pipeline with Euclidean distance (Method 1) is the most effective approach tested. Future work should explore whether a combined representation with a more carefully weighted distance — rather than a flat Generalised Gower distance — could recover additional structure from the clinical modality.